### Volatilidade 12 Meses do IPCA

*Mede o risco inflacionário ao longo do tempo.*

A volatilidade é calculada como o desvio padrão do IPCA dentro de um grupo (por exemplo, por cenário) nos últimos 12 meses:

$$
\text{Vol}_{IPCA,12m} = \sigma(\pi^e_{\text{últimos 12 meses}})
$$

Onde:
- \(\pi^e\) = inflação esperada (IPCA)  
- \(\sigma\) = desvio padrão  

Essa métrica ajuda a entender quão **instável** está a inflação em cada cenário, permitindo avaliar o **risco inflacionário**.

In [ ]:
# Cria uma cópia do dataframe com os cálculos de volatilidade do juros real
df_volatil_12m_ipca = df_volatil_juros_real

# Filtra apenas linhas do IPCA
df_ipca_vol_12m = df_volatil_12m_ipca[df_volatil_12m_ipca["indicador"] == "IPCA"].copy()

In [ ]:
# Calcula Volatilidade 12 meses por cenário
# rolling de 12 períodos (12 meses)
df_ipca_vol_12m["vol_ipca_12m"] = (
    df_ipca_vol_12m.groupby("cenarios")["vl_taxa"]
            .transform(lambda x: x.rolling(12, min_periods=1).std())
)

In [ ]:
# Integra de volta ao dataframe original
# mantém todas as linhas (SELIC e IPCA)
df_volatil_12m_ipca = df_volatil_12m_ipca.merge(
    df_ipca_vol_12m[["data","cenarios","vol_ipca_12m"]],
    on=["data","cenarios"],
    how="left"
)

### Média da SELIC no Período Projetado
Resumo comparativo de cenários.

A média da SELIC é calculada exclusivamente para as observações marcadas como **projetado** no horizonte entre 2026 e 2030, desconsiderando os dados históricos anteriores.

Se \( r_t \) for a taxa mensal da SELIC e \( T \) o conjunto de meses com `periodo_cenario = 'projetado'`, a média é dada por:

$$
\bar{r}_{data projetada} =
\frac{1}{N}
\sum_{t \in T} r_t
$$

Onde:
- \( r_t \) = taxa SELIC no mês \( t \) dentro do período projetado  
- \( T \) = conjunto de meses entre jan/2026 e dez/2030 com `periodo_cenario = 'projetado'`  
- \( N \) = número total de meses considerados no período  
- \( \sum \) = soma das taxas dentro do intervalo selecionado  

> Essa métrica representa a postura média da política monetária no horizonte projetado, considerando apenas os dados classificados como projeção.



In [ ]:
# Cria uma cópia do dataframe com os cálculos da media do periodo 
df_media_periodo_projetado = df_acum_12m_ipca.copy()

# Filtrar linhas SELIC e período projetado
df_selic_proj = df_media_periodo_projetado[
    (df_media_periodo_projetado['indicador'] == 'SELIC') & 
    (df_media_periodo_projetado['periodo_cenario'] == 'projetado')
]

In [ ]:
# Calcular média da SELIC somente no período projetado
media_selic_proj = df_selic_proj['vl_taxa'].mean()

# Criar coluna nova com NaN
df_media_periodo_projetado['media_selic_projetada'] = np.nan

# Preencher apenas para datas a partir de jan/2026
df_media_periodo_projetado.loc[
    (df_media_periodo_projetado['periodo_cenario'] == 'projetado') & 
    (df_media_periodo_projetado['indicador'] == 'SELIC'),
    'media_selic_projetada'
] = media_selic_proj

df_media_periodo_projetado = df_media_periodo_projetado.rename(columns={'data': 'data_projetada'})

# Tranformações necessárias para o agente interpretador

## Separar os Dataframes

In [ ]:
# Filtra apenas os dados observados (histórico)
df_observado = df_media_periodo_projetado[df_media_periodo_projetado['periodo_cenario'] == 'observado'].copy()

# Filtra apenas os dados projetados (cenários futuros)
df_projetado = df_media_periodo_projetado[df_media_periodo_projetado['periodo_cenario'] == 'projetado'].copy()

## Pivot separado

In [ ]:
# Função para pivotar e consolidar métricas
def pivotar_base(df_base):
    """
    Esta função transforma a base no formato analítico ideal
    para o agente de IA.

    Ela:
    - Pivotará IPCA e SELIC para ficarem na mesma linha
    - Consolidará métricas econômicas por data e cenário
    """
    # Pivotar os valores das taxas (IPCA e SELIC)
    # Resultado: IPCA e SELIC viram colunas
    valores = df_base.pivot_table(
        index=['data_projetada', 'cenarios'],   # Agrupamento por data e cenário
        columns='indicador',                   # IPCA e SELIC viram colunas
        values='vl_taxa'                       # Valores das taxas
    ).reset_index()

    # Consolidar métricas que já são iguais dentro do grupo
    # (juros_real e volatilidades)
    metricas = df_base.groupby(
        ['data_projetada', 'cenarios']
    ).agg({
        'juros_real': 'first',                     # Mesmo valor dentro do grupo
        'vol_juros_real': 'first',
        'vol_juros_real_ano_cenario': 'first',
        'vol_juros_real_rolante_3m': 'first',

        # Essas existem apenas para um indicador,
        # usamos 'max' para evitar perder o valor e eliminar NaN estrutural
        'vol_ipca_12m': 'max',
        'ipca_acum_12m': 'max',
        'media_selic_projetada': 'max'
    }).reset_index()
    
    # Unir valores pivotados com métricas consolidadas
    df_final = valores.merge(
        metricas,
        on=['data_projetada', 'cenarios'],
        how='left'
    )
    return df_final

In [ ]:
# Aplicar a função nas duas bases
df_obs_final = pivotar_base(df_observado)
df_proj_final = pivotar_base(df_projetado)

# Motor Temporal
## Análise Macroeconômica Microdinâmica por Cenário

Este notebook implementa o Motor Temporal do Agente Macroeconômico.

Objetivo:
Capturar a dinâmica mensal das variáveis macroeconômicas por cenário.

Aplicações:
- Tendência linear
- Volatilidade rolante
- Correlação rolante
- Z-score
- IQR
- Identificação de anomalias
- Consolidação estatística por cenário

Output:
Arquivo estruturado contendo os resultados temporais por cenário.


### Tendência Mensal (Regressão Linear)
**Identificar se há tendência de alta ou queda ao longo do tempo.**

Para cada cenário:

**Yₜ = β0 + β1t + εt**

- **yt** → valor do indicador no mês 𝑡
- **t** → número do mês (1, 2, 3, …)
- **β0** → intercepto
- **β1** → coeficiente angular, que representa a tendência mensal
  
Se:
- **β > 0** → tendência de alta  
- **β < 0** → tendência de queda  
- **β ≈ 0** → estabilidade
- **𝜀𝑡**  → erro residual

In [224]:
def calcular_tendencia(grupo, coluna):
    """
    Calcula a tendência linear de uma coluna usando regressão linear.
    
    Parâmetros:
    grupo : DataFrame -> grupo de dados (por cenário)
    coluna : str -> coluna numérica (ex: 'IPCA' ou 'SELIC')
    
    Retorna:
    beta : float -> coeficiente de inclinação da reta (tendência)
    """
    # Remove valores ausentes da coluna
    grupo = grupo.dropna(subset=[coluna])
    
    # Se houver menos de 2 pontos, não é possível calcular tendência
    if len(grupo) < 2:
        return np.nan
    
    # X = vetor temporal (0,1,2,...)
    X = np.arange(len(grupo)).reshape(-1,1)
    
    # y = valores da coluna
    y = grupo[coluna].values
    
    # Ajusta modelo de regressão linear
    modelo = LinearRegression()
    modelo.fit(X, y)
    
    # Retorna o coeficiente beta (inclinação)
    return modelo.coef_[0]


In [226]:
# Aplicação por cenário
tendencias = []

for cenario, grupo in df_temporal.groupby('cenarios'):
    # Calcula beta de IPCA
    beta_ipca = calcular_tendencia(grupo, 'IPCA')
    
    # Calcula beta de SELIC
    beta_selic = calcular_tendencia(grupo, 'SELIC')
    
    # Armazena resultados em lista de dicionários
    tendencias.append({
        'cenario': cenario,
        'beta_ipca': beta_ipca,
        'beta_selic': beta_selic
    })

# Converte lista em DataFrame final de tendências
df_tendencias = pd.DataFrame(tendencias)

# Rolling Volatility

### Volatilidade móvel (ex: janela de 3 meses)

*medir instabilidade recente em IPCA e SELIC.*

**σ_t = std(r_{t-n+1}, ..., r_t)**

Onde:

- r_t = retorno no período t
- n = tamanho da janela (ex: 3 meses)
- σ_t = volatilidade no período t

Interpretação:

- σ_t alto  → maior risco / maior variabilidade
- σ_t baixo → menor risco / maior estabilidade

In [227]:
def rolling_volatilidade(df, coluna, janela=3, group_col='cenarios', data_col='data_projetada'):
    """
    Calcula volatilidade móvel (desvio padrão) de uma coluna, respeitando ordem temporal e cenário.
    
    Parâmetros:
    df : DataFrame base
    coluna : str -> coluna numérica (ex: 'IPCA')
    janela : int -> tamanho da janela móvel (default=3)
    group_col : str -> coluna para agrupar (ex: 'cenarios')
    data_col : str -> coluna temporal para ordenar
    
    Retorna:
    Series com volatilidade rolling alinhada ao índice original
    """
    # Ordena dataframe por cenário e data
    df_ordenado = df.sort_values([group_col, data_col])
    
    # Calcula rolling std para cada grupo
    rolling_vol = (
        df_ordenado
            .groupby(group_col)[coluna]
            .transform(lambda x: x.rolling(janela).std())
    )
    
    return rolling_vol

In [228]:
# Aplicação da função rolling_volatilidade
df_temporal['vol_ipca_3m'] = rolling_volatilidade(df_temporal, 'IPCA', janela=3)
df_temporal['vol_selic_3m'] = rolling_volatilidade(df_temporal, 'SELIC', janela=3)

# Rolling Correlation (IPCA vs SELIC)

### Correlação móvel entre IPCA e SELIC

A rolling correlation mede como a relação linear entre duas variáveis
(IPCA e SELIC) evolui ao longo do tempo, utilizando uma janela deslizante
de tamanho fixo.

Para cada período t, calcula-se a correlação de Pearson
entre as últimas n observações:

**ρ_t = corr(X_{t-n+1}, ..., X_t ; Y_{t-n+1}, ..., Y_t)**

Onde:

- X_t = variação do IPCA no período t  
- Y_t = variação da SELIC no período t  
- n = tamanho da janela  
- ρ_t = correlação estimada no período t  

Interpretação:

- ρ_t ≈ +1 → forte relação linear positiva  
- ρ_t ≈ -1 → forte relação linear negativa  
- ρ_t ≈ 0  → ausência de relação linear  

Observação:
A rolling correlation permite identificar mudanças dinâmicas
na relação entre inflação e política monetária ao longo do tempo.

In [205]:
def rolling_correlacao(df, coluna_x, coluna_y, janela=6, group_col='cenarios', data_col='data_projetada'):
    """
    Calcula correlação móvel entre duas variáveis por grupo (cenário).
    
    Parâmetros:
    df : DataFrame base
    coluna_x : str -> primeira variável (ex: 'IPCA')
    coluna_y : str -> segunda variável (ex: 'SELIC')
    janela : int -> tamanho da janela móvel (default=6)
    group_col : str -> coluna para agrupar (cenários)
    data_col : str -> coluna temporal para ordenar
    
    Retorna:
    Series com correlação móvel alinhada ao índice original
    """
    # Ordena o dataframe por cenário e data
    df_ordenado = df.sort_values([group_col, data_col])
    
    # Cria Series vazia para armazenar resultados
    resultado = pd.Series(index=df_ordenado.index, dtype=float)
    
    # Calcula correlação rolling por grupo
    for _, grupo in df_ordenado.groupby(group_col):
        corr = grupo[coluna_x].rolling(janela).corr(grupo[coluna_y])
        resultado.loc[grupo.index] = corr
    
    return resultado

In [229]:
# Aplicação da função rolling_correlacao
df_temporal['corr_6m'] = rolling_correlacao(df_temporal, 'IPCA', 'SELIC', janela=6)

# Z-Score (Detecção de Anomalias)

O Z-Score mede quantos desvios-padrão uma observação está
acima ou abaixo da média.

Fórmula:

$$
Z = \frac{x - \mu}{\sigma}
$$

Onde:

- x  = valor observado  
- μ  = média da série  
- σ  = desvio padrão da série  

Interpretação:

- |Z| > 2 → possível anomalia  
- |Z| > 3 → forte evidência de outlier  
- Z ≈ 0   → valor próximo da média  

In [231]:
# Calcula z-score de IPCA e SELIC por cenário
df_temporal['z_ipca'] = df_temporal.groupby('cenarios')['IPCA'].transform(
    lambda x: (x - x.mean()) / x.std()  # Desvio em relação à média do grupo
)
df_temporal['z_selic'] = df_temporal.groupby('cenarios')['SELIC'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# IQR (Outliers Estruturais)

O IQR (Interquartile Range) é uma medida de dispersão baseada
nos quartis da distribuição e é amplamente utilizado para
identificar outliers estruturais.

Definição:

**IQR = Q3 - Q1**

Onde:

- Q1 = primeiro quartil (25%)
- Q3 = terceiro quartil (75%)
- IQR = intervalo interquartil

Critério para detecção de outliers:

- Limite inferior = Q1 - 1.5 × IQR
- Limite superior = Q3 + 1.5 × IQR

Interpretação:

- Valores abaixo do limite inferior → possível outlier inferior
- Valores acima do limite superior → possível outlier superior

Observação:

O método do IQR é robusto a valores extremos,
pois não depende da média nem do desvio padrão.

In [232]:
def flag_iqr_series(x):
    """
    Identifica outliers estruturais usando Interquartile Range (IQR)
    
    Retorna:
    Series booleana indicando True para outliers
    """
    Q1 = x.quantile(0.25)  # 1º quartil
    Q3 = x.quantile(0.75)  # 3º quartil
    IQR = Q3 - Q1           # Intervalo interquartil
    
    # Retorna True para valores fora do intervalo [Q1-1.5*IQR, Q3+1.5*IQR]
    return (x < Q1 - 1.5*IQR) | (x > Q3 + 1.5*IQR)

In [233]:
# Aplicação da função flag_iqr_series
df_temporal['outlier_ipca'] = df_temporal.groupby('cenarios')['IPCA'].transform(flag_iqr_series)